In [1]:
from langgraph.graph import StateGraph , START , END
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint

from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict

In [2]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    provider="auto",
    temperature=0.7,
    max_new_tokens=2048
)

model = ChatHuggingFace(llm = llm)

c:\LangChain and LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class LLMState(TypedDict):
    question:str
    answer:str

In [11]:
def llm_qa(state:LLMState)->LLMState:
    question = state['question']

    prompt = f'Answer the following question {question} in one line'

    answer = model.invoke(prompt).content

    state['answer'] = answer

    return state

In [12]:
#create graph
graph = StateGraph(LLMState)

#add nodes
graph.add_node('llm_qa',llm_qa)

#add edges
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

#compile
workflow = graph.compile()

In [13]:
initial_state = {'question':'Average distance of moon from the earth?'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'Average distance of moon from the earth?', 'answer': 'The Moon’s average distance from Earth is about\u202f384,400\u202fkilometers (≈238,855\u202fmiles).'}
